# 03 - Leakage-Safe Feature Engineering

The single production code path `FeatureEngine` (`src/feature_engineering.py`)
computes every feature **as-of** the transaction: history before the row is
allowed, the row itself is not. Priors and segment baselines are fitted on
training data only.


In [ ]:
import os, sys
ROOT = os.path.dirname(os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


In [ ]:
from src.config import get_settings
from src.data_loader import load_dataframe, valid_numeric
from src.preprocessing import time_split, unseen_customer_split
from src.feature_engineering import (FeatureEngine, MODEL_FEATURES,
                                     NOVELTY_FEATURES)

cfg = get_settings()
df = valid_numeric(load_dataframe(cfg)[0])
train, val, test = time_split(df, cfg)
train, val, test, heldout = unseen_customer_split(train, val, test, df, cfg)
print(f"train={len(train):,} val={len(val):,} test={len(test):,}")
print(f"unseen-customer holdout removed from train: {len(heldout):,}")


In [ ]:
priors_engine = FeatureEngine(cfg)
priors = priors_engine.compute_priors(train)
print("priors fitted on:", priors['fitted_on'])
print("global log-amount prior mean/std:", round(priors['global']['mean'], 3),
      round(priors['global']['std'], 3))


In [ ]:
import pandas as pd
combined = pd.concat([train, val, test], ignore_index=True).sort_values(
    ["ts_sec", "transaction_id"]).reset_index(drop=True)
engine = FeatureEngine(cfg, priors)
feats = engine.replay(combined, with_parity=True)

parity_v = float(feats['velocity_8min'].eq(feats['_parity_velocity_8min']).mean())
parity_s = float(feats['amount_spike'].eq(feats['_parity_amount_spike']).mean())
print(f"parity velocity_8min : {parity_v:.4f}")
print(f"parity amount_spike  : {parity_s:.4f}")
print(f"model features       : {len(MODEL_FEATURES)}  "
      f"novelty features: {len(NOVELTY_FEATURES)}")


In [ ]:
cold = feats[feats['customer_history_count'] == 0]
est  = feats[feats['customer_history_count'] >= 5]
print("cold-start share: {:.1f}%  (n={:,})".format(
    100*len(cold)/len(feats), len(cold)))
print("mean |amount_z_shrunk|  cold: {:.2f} | established: {:.2f}".format(
    cold['amount_z_shrunk'].abs().mean(), est['amount_z_shrunk'].abs().mean()))


## Design rules enforced
1. Strict chronological processing; `prepare_row` before `commit_row`.
2. Raw `customer_id/merchant_id/device_id` never become features – they only
   key internal state.
3. Segment priors, amount bins and global baselines are frozen from training.
4. Cold-start baselines shrink thin customer history toward the segment prior
   (`kappa`), then the global prior – missing history is not treated as fraud.
5. Parity columns (`velocity_8min`, `amount_spike`) are recomputed from first
   principles and compared with the provided ones to prove point-in-time
   correctness (>>99% match).
